# Part 2: Retrieval-Augmented Generation with the Part 1 SLM

This notebook reloads **Architecture / Model #3** selected in `assignment_P1-A7_SLM.ipynb` from the local `tiny_transformer.pt` checkpoint. It compares the same five author questions without retrieval and with RAG, using the same character vocabulary, model weights, compact question cues, seed, and generation settings.

The separate `knowledge-base-documents` folder contains five JSON **source documents**, one per category from [Kirkus Reviews' Best of 2025 lists](https://www.kirkusreviews.com/best-of/2025/):

1. Best Mysteries and Thrillers of 2025
2. Best Book Club Fiction of 2025
3. Best Romance of 2025
4. Best Dystopian Fiction of 2025
5. Best Historical Fiction of 2025

The 45 individual book records become uniquely identified **retrieval documents**. Top-K counts retrieved book records, with K = 1, 2, 3, and 5. TF-IDF searches their original text; only the selected author evidence and shared question cue are normalized to the SLM vocabulary and packed into its 32-character window. Compatibility checks audit every actual prompt, and inference never updates the weights.

In [1]:
# CELL 1: Load the selected Part 1 SLM from tiny_transformer.pt
from pathlib import Path
import json
import random
import time
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
DEVICE = torch.device("cpu")
TEMPERATURE = 0.7
MAX_NEW_CHARACTERS = 80
TOP_K_VALUES = (1, 2, 3, 5)
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

roots = [Path.cwd(), *Path.cwd().parents]
candidates = roots + [root / "Assignment_P1-A7" / "moveon" / "6-moveon" / "Assignment_P1-A7" for root in roots]
ASSIGNMENT_DIR = next((path for path in candidates if (path / "tiny_transformer.pt").is_file() and (path / "knowledge-base-documents").is_dir()), None)
if ASSIGNMENT_DIR is None:
    raise FileNotFoundError("Cannot locate tiny_transformer.pt beside knowledge-base-documents.")
CHECKPOINT_PATH = ASSIGNMENT_DIR / "tiny_transformer.pt"
KB_DIR = ASSIGNMENT_DIR / "knowledge-base-documents"
CORPUS_PATH = ASSIGNMENT_DIR / "training_corpus.txt"

# Match the architecture used to save the selected Part 1 checkpoint.
class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, feedforward_dim):
        super().__init__()
        self.attention = nn.MultiheadAttention(embedding_dim, num_heads, dropout=0.0, batch_first=True)
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.feedforward = nn.Sequential(
            nn.Linear(embedding_dim, feedforward_dim),
            nn.ReLU(),
            nn.Linear(feedforward_dim, embedding_dim)
        )

    def forward(self, x):
        length = x.shape[1]
        mask = torch.triu(torch.ones(length, length, device=x.device, dtype=torch.bool), diagonal=1)
        normalized = self.ln1(x)
        attended, _ = self.attention(normalized, normalized, normalized, attn_mask=mask, need_weights=False)
        x = x + attended
        return x + self.feedforward(self.ln2(x))

class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_heads, num_layers, feedforward_dim, context_length):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(context_length, embedding_dim)
        self.blocks = nn.Sequential(*[
            TransformerBlock(embedding_dim, num_heads, feedforward_dim)
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(embedding_dim)
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, input_ids):
        _, length = input_ids.shape
        if length > self.context_length:
            raise ValueError("Input sequence exceeds context length.")
        positions = torch.arange(length, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.blocks(x)
        x = self.ln_final(x)
        return self.output(x)

saved = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
config = saved["config"]
chars = saved["vocab"]
stoi = {char: index for index, char in enumerate(chars)}
itos = {index: char for index, char in enumerate(chars)}
model = TinyTransformerLM(vocab_size=len(chars), **config).to(DEVICE)
model.load_state_dict(saved["model_state"], strict=True)
model.eval()
checkpoint_weights = {name: tensor.detach().clone() for name, tensor in model.state_dict().items()}
CONTEXT_LENGTH = model.context_length
print("Loaded:", CHECKPOINT_PATH)
print("Architecture:", config)
print("Vocabulary:", len(chars), "characters")
print("Context:", CONTEXT_LENGTH, "characters")

Loaded: C:\Users\chant\Desktop\UIUC\INFO 490 - Advanced AI Web App Development\Modules\Week 7\Week 7 Scripts\Week 7 Scripts\Assignment_P1-A7\moveon\6-moveon\Assignment_P1-A7\tiny_transformer.pt
Architecture: {'embedding_dim': 48, 'num_heads': 4, 'num_layers': 1, 'feedforward_dim': 192, 'context_length': 32}
Vocabulary: 27 characters
Context: 32 characters


## Context length and tokenizer compatibility

The checkpoint defines a **32-character context window** and a **27-character vocabulary**; one character is one token. The code checks those values against the loaded embedding layers and output layer.

Both conditions use the same compact `<requested title> by` question cue. Text intended for the SLM is lowercased, Unicode accents are reduced to base letters, and any remaining unsupported characters are replaced with spaces; the normalization audit records these changes. Original JSON text stays intact for TF-IDF retrieval and display, and the model vocabulary is not expanded.

Prompts are budgeted **before** generation: the complete question cue is reserved first, then the remaining characters are shared across retrieved author facts. Each shortened name is recorded explicitly. The generator rejects any prompt longer than the configured window or containing an unsupported character; it does not silently crop or normalize inputs. During generation, each forward pass uses at most the latest 32 characters, so old evidence eventually leaves the window.

In [2]:
import unicodedata

# Validate the tokenizer and context against the actual loaded checkpoint.
if len(chars) != len(set(chars)) or any(len(char) != 1 for char in chars):
    raise ValueError("The checkpoint must contain a unique character vocabulary.")
if " " not in stoi or "\n" not in stoi:
    raise ValueError("The saved vocabulary must support the prompt separators.")
assert CONTEXT_LENGTH == model.position_embedding.num_embeddings
assert len(chars) == model.token_embedding.num_embeddings == model.output.out_features
model.requires_grad_(False)

def normalize_for_slm(text):
    original = str(text)
    lowered = original.lower()
    decomposed = unicodedata.normalize("NFKD", lowered)
    base_letters = "".join(char for char in decomposed if not unicodedata.combining(char))
    unsupported = sorted(set(base_letters) - set(stoi))
    normalized = "".join(char if char in stoi else " " for char in base_letters)
    audit = {
        "original_text": original,
        "normalized_text": normalized,
        "case_changed": original != lowered,
        "unicode_changed": base_letters != lowered,
        "unsupported_replaced": [repr(char) for char in unsupported],
        "replacement_count": sum(char not in stoi for char in base_letters),
    }
    assert set(normalized) <= set(stoi)
    return normalized, audit

def encode_slm_prompt(prompt):
    # Strict encoding: normalization and deliberate truncation happen earlier.
    if not isinstance(prompt, str) or not prompt:
        raise ValueError("The model prompt must be a nonempty string.")
    unsupported = sorted(set(prompt) - set(stoi))
    if unsupported:
        raise ValueError(f"Prompt contains unsupported characters: {unsupported!r}")
    if len(prompt) > CONTEXT_LENGTH:
        raise ValueError(f"Prompt has {len(prompt)} characters; limit is {CONTEXT_LENGTH}.")
    token_ids = [stoi[char] for char in prompt]
    assert "".join(itos[token_id] for token_id in token_ids) == prompt
    return token_ids

def compact_author_question(question):
    if "Who wrote " not in question:
        raise ValueError("Expected an author question containing 'Who wrote '.")
    requested_title = question.split("Who wrote ", 1)[1].rstrip(" ?")
    cue, normalization = normalize_for_slm(requested_title + " by")
    encode_slm_prompt(cue)  # Reject an oversized cue instead of dropping the question.
    return cue, normalization

@torch.no_grad()
def generate_slm(prompt, max_new_characters=MAX_NEW_CHARACTERS,
                 temperature=TEMPERATURE, seed=SEED):
    if temperature <= 0 or max_new_characters < 1:
        raise ValueError("Use a positive temperature and generation length.")
    token_ids = encode_slm_prompt(prompt)
    ids = torch.tensor([token_ids], dtype=torch.long, device=DEVICE)
    torch.manual_seed(seed)
    model.eval()
    generated = []
    max_context_seen = 0
    for _ in range(max_new_characters):
        # Rolling context: every actual model input stays within the checkpoint limit.
        context = ids[:, -CONTEXT_LENGTH:]
        assert 1 <= context.shape[1] <= CONTEXT_LENGTH
        max_context_seen = max(max_context_seen, context.shape[1])
        logits = model(context)[:, -1, :] / temperature
        assert logits.shape[-1] == len(chars)
        next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        ids = torch.cat((ids, next_id), dim=1)
        generated.append(itos[next_id.item()])
    response = "".join(generated).strip()
    assert set(response) <= set(stoi)
    return {
        "response": response,
        "visible_prompt": prompt,
        "prompt_characters": len(prompt),
        "vocabulary_valid": True,
        "max_model_input_characters": max_context_seen,
    }

print("Saved vocabulary:", repr(chars))
print("Configured context:", CONTEXT_LENGTH, "characters")
print("Strict prompt validation is enabled; initial prompts are never silently cropped.")

Saved vocabulary: ['\n', ' ', '.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y']
Configured context: 32 characters
Strict prompt validation is enabled; initial prompts are never silently cropped.


## Five questions for the baseline and RAG tests

Each question asks for one verifiable author fact. Both conditions encode it as the **same normalized compact cue**, `<requested title> by`, so the requested title and author task remain visible; RAG adds retrieved author text before that cue.

1. **Q1 — Mystery and thrillers:** Who wrote *King Of Ashes*?
2. **Q2 — Book-club fiction:** Who wrote *Flashlight*?
3. **Q3 — Dystopian fiction:** Who wrote *The Dream Hotel*?
4. **Q4 — Romance:** Who wrote *Unloved*?
5. **Q5 — Historical fiction:** Who wrote *Isola*?

The full question is used for retrieval. The original category label and question wording are omitted from both generator conditions to fit the context window; the compact cue is preserved without truncation. Correct authors from the source files are used only for RAG evidence and later evaluation, never for baseline prompts.

## Test the SLM Without RAG

The saved SLM receives only each compact question cue, with **no retrieved evidence**. Both conditions use temperature **0.7**, seed **42**, and **80 new characters**; the shared generator accepts only validated prompts and applies a rolling 32-character window during generation.

| Question | SLM Input Without RAG (characters/limit) | SLM Response Without RAG |
|---|---|---|
| **Q1** — Mystery: Who wrote King Of Ashes? | `"king of ashes by"` (16/32) | stely offers courses in informatics.<br>students experiment with language models.<br>a |
| **Q2** — Book club: Who wrote Flashlight? | `"flashlight by"` (13/32) | utseful next token.<br>ork.<br>inference uses previously learned parameters.<br>the univ |
| **Q3** — Dystopian: Who wrote The Dream Hotel? | `"the dream hotel by"` (18/32) | weith language models.<br>artificial intelligence can generate text.<br>the model pre |
| **Q4** — Romance: Who wrote Unloved? | `"unloved by"` (10/32) | s deviews.<br>training changes the weights of a neural network.<br>inference uses prev |
| **Q5** — Historical: Who wrote Isola? | `"isola by"` (8/32) | the ts use attention mechanisms.<br>neural networks learn from training data.<br>machi |

All five initial prompts use only the saved vocabulary and fit the configured window. None of these responses names the requested author; incomplete endings are retained from the generated text.

In [3]:
# Baseline: use only the same compact question cue later used by RAG.
BASELINE_QUESTIONS = [
    ("Q1", "Mystery: Who wrote King Of Ashes?"),
    ("Q2", "Book club: Who wrote Flashlight?"),
    ("Q3", "Dystopian: Who wrote The Dream Hotel?"),
    ("Q4", "Romance: Who wrote Unloved?"),
    ("Q5", "Historical: Who wrote Isola?"),
]

baseline_rows = []
for question_id, question in BASELINE_QUESTIONS:
    question_cue, normalization = compact_author_question(question)
    result = generate_slm(question_cue)
    baseline_rows.append({
        "Question": question_id,
        "Question Text": question,
        "Question Cue": question_cue,
        "Question Normalization": normalization,
        "SLM Response Without RAG": result["response"],
        "Visible Prompt": result["visible_prompt"],
        "Prompt Characters": result["prompt_characters"],
        "Vocabulary Valid": result["vocabulary_valid"],
        "Max Model Input Characters": result["max_model_input_characters"],
    })

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df[["Question", "Question Text", "Visible Prompt",
                     "Prompt Characters", "SLM Response Without RAG"]])
print("All five baseline prompts passed strict vocabulary and context checks.")

,Question,Question Text,Visible Prompt,Prompt Characters,SLM Response Without RAG
0,Q1,Mystery: Who wrote King Of Ashes?,king of ashes by,16,stely offers courses in informatics.\nstudents...
1,Q2,Book club: Who wrote Flashlight?,flashlight by,13,utseful next token.\nork.\ninference uses prev...
2,Q3,Dystopian: Who wrote The Dream Hotel?,the dream hotel by,18,weith language models.\nartificial intelligenc...
3,Q4,Romance: Who wrote Unloved?,unloved by,10,s deviews.\ntraining changes the weights of a ...
4,Q5,Historical: Who wrote Isola?,isola by,8,the ts use attention mechanisms.\nneural netwo...


All five baseline prompts passed strict vocabulary and context checks.


## Applying RAG to the SLM

The **five JSON files are source documents**, and their **45 book records are retrieval documents**. Top-K counts retrieved book records. TF-IDF and cosine similarity use the full author question to rank records;


For generation, the complete compact question cue is reserved first. The remaining 32-character budget is divided among normalized author facts, with the best-ranked fact nearest the cue. Original record IDs and titles are shown for auditing; **only the visible prompt is sent to the SLM**. At higher K, author names can be shortened and the other books' title associations are omitted.

In the prompt column, `\n` represents one newline and the count includes every prompt character. Normalization changes and deliberate name truncation are recorded separately in the code and compatibility audit; generation performs no hidden initial cropping.

| Question | Top-K | Retrieved Book Records | Visible Prompt (characters/limit) | Author Truncated | Generated Response | Response Quality (1–10) |
|---|---:|---|---|---|---|---:|
| Q1 | 1 | 01_02 King Of Ashes | `"s.a. cosby\nking of ashes by"` (27/32) | No | stems.<br>language models predict the next token.<br>transformers use attention mechan | 1 |
| Q1 | 2 | 01_02 King Of Ashes; 01_08 Buried Above Ground | `"mike ri s.a. co\nking of ashes by"` (32/32) | Yes | stems.<br>language models predict the next token.<br>transformers use attention mechan | 1 |
| Q1 | 3 | 01_02 King Of Ashes; 01_08 Buried Above Ground; 01_05 Detective Aunty | `"u ma mike s.a. \nking of ashes by"` (32/32) | Yes | stems.<br>language models predict the next token.<br>transformers use attention mechan | 1 |
| Q1 | 5 | 01_02 King Of Ashes; 01_08 Buried Above Ground; 01_05 Detective Aunty; 04_10 Ladies In Hating; 03_05 What We Can Know | `"ia al u  mi s.a\nking of ashes by"` (32/32) | Yes | stems.<br>language models predict the next token.<br>transformers use attention mechan | 1 |
| Q2 | 1 | 02_01 Flashlight | `"susan choi\nflashlight by"` (24/32) | No | uts th ton programming.<br>students develop artificial intelligence systems.<br>langu | 1 |
| Q2 | 2 | 02_01 Flashlight; 02_08 Gaysians | `"mike cur susan cho\nflashlight by"` (32/32) | Yes | utse ursing probabilities.<br>attention helps models use contextual information.<br>p | 1 |
| Q2 | 3 | 02_01 Flashlight; 02_08 Gaysians; 02_07 Moderation | `"elain mike  susan \nflashlight by"` (32/32) | Yes | utse ursing probabilities.<br>attention helps models use contextual information.<br>p | 1 |
| Q2 | 5 | 02_01 Flashlight; 02_08 Gaysians; 02_07 Moderation; 02_05 Fresh, Green Life; 03_05 What We Can Know | `"ia seb ela mik sus\nflashlight by"` (32/32) | Yes | utse ursing probabilities.<br>attention helps models use contextual information.<br>p | 1 |
| Q3 | 1 | 03_03 The Dream Hotel | `"laila lalami\nthe dream hotel by"` (31/32) | No | utses use models and views.<br>training changes the weights of a neural network.<br>i | 1 |
| Q3 | 2 | 03_03 The Dream Hotel; 05_08 The Listeners | `"maggie laila \nthe dream hotel by"` (32/32) | Yes | utses use models and views.<br>training changes the weights of a neural network.<br>i | 1 |
| Q3 | 3 | 03_03 The Dream Hotel; 05_08 The Listeners; 03_02 Brother Brontë | `"fer magg lail\nthe dream hotel by"` (32/32) | Yes | utses use models and views.<br>training changes the weights of a neural network.<br>i | 1 |
| Q3 | 5 | 03_03 The Dream Hotel; 05_08 The Listeners; 03_02 Brother Brontë; 03_01 The Book Of Guilt; 03_07 Sacrament | `"s ca fe ma la\nthe dream hotel by"` (32/32) | Yes | utses use models and views.<br>training changes the weights of a neural network.<br>i | 1 |
| Q4 | 1 | 04_01 Unloved | `"peyton corinne\nunloved by"` (25/32) | No | s learned parameters.<br>the university offers courses in informatics.<br>students exp | 1 |
| Q4 | 2 | 04_01 Unloved; 04_02 Chaos | `"constance  peyton cor\nunloved by"` (32/32) | Yes | s learned parameters.<br>the university offers courses in informatics.<br>students exp | 1 |
| Q4 | 3 | 04_01 Unloved; 04_02 Chaos; 04_09 Can'T Get Enough | `"kenned consta peyton \nunloved by"` (32/32) | Yes | s learned parameters.<br>the university offers courses in informatics.<br>students exp | 1 |
| Q4 | 5 | 04_01 Unloved; 04_02 Chaos; 04_09 Can'T Get Enough; 04_05 It Had To Be Him; 04_04 A Tropical Rebel Gets The Duke | `"adr adi ken cons peyt\nunloved by"` (32/32) | Yes | s learned parameters.<br>the university offers courses in informatics.<br>students exp | 1 |
| Q5 | 1 | 05_01 Isola | `"allegra goodman\nisola by"` (24/32) | No | the ts use nexperiment with language models.<br>artificial intelligence can generat | 1 |
| Q5 | 2 | 05_01 Isola; 05_07 The Antidote | `"karen russe allegra goo\nisola by"` (32/32) | Yes | the university offers courses in informatics.<br>students experiment with language | 1 |
| Q5 | 3 | 05_01 Isola; 05_07 The Antidote; 05_04 The Buffalo Hunter Hunter | `"stephen karen r allegra\nisola by"` (32/32) | Yes | the tstudents are building useful software products.<br>small language models can r | 1 |
| Q5 | 5 | 05_01 Isola; 05_07 The Antidote; 05_04 The Buffalo Hunter Hunter; 05_05 Fonseca; 05_08 The Listeners | `"mag jess step kare alle\nisola by"` (32/32) | Yes | the university offers courses in informatics.<br>students experiment with language | 1 |

**Quality assessment:** Top-1 retrieves the intended book and retains its complete normalized author for each question, yet none of the 20 continuations answers the author question. All are rated **1/10** for factual accuracy and relevance. Compatible inputs establish that the experiment runs as configured; they do not guarantee that the SLM can use the supplied fact.

In [4]:
# Five source JSON files -> separately identified book retrieval documents.
# Retrieval uses the original Unicode text; only text sent to the SLM is normalized.
book_records = []
source_documents = []
seen_source_ids = set()
seen_book_ids = set()
for path in sorted(KB_DIR.glob("*.json")):
    with path.open("r", encoding="utf-8") as handle:
        source = json.load(handle)
    source_id = str(source["document_id"])
    if source_id in seen_source_ids:
        raise ValueError(f"Duplicate source document: {source_id}")
    seen_source_ids.add(source_id)
    source_title = str(source["document_title"])
    source_documents.append({"id": source_id, "title": source_title, "file": path.name})
    for book in source["books"]:
        book_id = str(book["book_id"])
        title = str(book["book_title"]).strip()
        author = str(book["author"]).strip()
        if book_id in seen_book_ids or not title or not author:
            raise ValueError(f"Invalid or duplicate book record: {book_id}")
        seen_book_ids.add(book_id)
        author_slm, author_normalization = normalize_for_slm(author)
        searchable = " ".join([
            title, title, source_title, author,
            str(book.get("genre", "")),
            str(book.get("synopsis", "")),
            str(book.get("review_synthesis", ""))
        ])
        book_records.append({
            "book_id": book_id, "source_document": source_id,
            "title": title, "author": author, "search_text": searchable,
            "author_slm": author_slm, "author_normalization": author_normalization
        })
if len(source_documents) != 5 or len(book_records) < max(TOP_K_VALUES):
    raise ValueError("Expected five source files with enough book records for Top-5.")
corpus_text = CORPUS_PATH.read_text(encoding="utf-8").lower()
assert all(book["search_text"].lower() not in corpus_text for book in book_records)

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
book_vectors = vectorizer.fit_transform([book["search_text"] for book in book_records])

def retrieve_books(question, k):
    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, book_vectors).ravel()
    ranked = sorted(range(len(book_records)),
                    key=lambda i: (-scores[i], book_records[i]["book_id"]))
    return [(book_records[i], float(scores[i])) for i in ranked[:k]]

def compact_book_prompt(question, retrieved):
    question_cue, _ = compact_author_question(question)
    suffix = "\n" + question_cue
    evidence_budget = CONTEXT_LENGTH - len(suffix)
    separators = len(retrieved) - 1
    if evidence_budget <= separators:
        raise ValueError("The full question cue leaves no room for evidence.")
    base, remainder = divmod(evidence_budget - separators, len(retrieved))
    fragments = []
    visible_evidence = []
    for rank, (book, _) in enumerate(retrieved):
        share = base + (rank < remainder)
        full_author = book["author_slm"]
        fragment = full_author[:share]  # Deliberate, audited evidence truncation.
        fragments.append(fragment)
        visible_evidence.append({
            "book_id": book["book_id"], "source_document": book["source_document"],
            "title": book["title"], "original_author": book["author"],
            "normalized_author": full_author, "visible_text": fragment,
            "omitted_text": full_author[len(fragment):],
            "author_truncated": len(fragment) < len(full_author),
            "normalization": book["author_normalization"],
        })
    prompt = " ".join(reversed(fragments)) + suffix
    encode_slm_prompt(prompt)  # Strict validation; generation will not crop it.
    assert prompt.endswith(question_cue)
    return prompt, visible_evidence, question_cue, evidence_budget

rag_rows = []
for question_id, question in BASELINE_QUESTIONS:
    for k in TOP_K_VALUES:
        retrieved = retrieve_books(question, k)
        prompt, evidence, question_cue, budget = compact_book_prompt(question, retrieved)
        baseline_cue = baseline_df.loc[baseline_df["Question"] == question_id, "Question Cue"].iloc[0]
        assert question_cue == baseline_cue
        result = generate_slm(prompt)
        assert result["visible_prompt"] == prompt
        rag_rows.append({
            "Question": question_id, "Top-K": k,
            "Retrieved Book IDs": [book["book_id"] for book, _ in retrieved],
            "Retrieved Titles": [book["title"] for book, _ in retrieved],
            "Source Documents": [book["source_document"] for book, _ in retrieved],
            "Similarity Scores": [round(score, 3) for _, score in retrieved],
            "Visible Evidence": evidence, "Question Cue": question_cue,
            "Evidence Budget": budget,
            "Omitted Evidence Characters": sum(len(item["omitted_text"]) for item in evidence),
            "Any Evidence Truncated": any(item["author_truncated"] for item in evidence),
            "Visible Prompt": result["visible_prompt"],
            "Prompt Characters": result["prompt_characters"],
            "Vocabulary Valid": result["vocabulary_valid"],
            "Max Model Input Characters": result["max_model_input_characters"],
            "Generated Response": result["response"],
        })

rag_df = pd.DataFrame(rag_rows)
assert len(rag_df) == len(BASELINE_QUESTIONS) * len(TOP_K_VALUES)
assert all(torch.equal(checkpoint_weights[name], tensor)
           for name, tensor in model.state_dict().items())
print("Indexed", len(book_records), "retrieval documents from", len(source_documents), "JSON files.")
print("All 20 RAG prompts passed strict vocabulary and context checks; weights are unchanged.")
display(rag_df[["Question", "Top-K", "Retrieved Book IDs", "Visible Prompt",
                "Prompt Characters", "Any Evidence Truncated", "Generated Response"]])

Indexed 45 retrieval documents from 5 JSON files.
All 20 RAG prompts passed strict vocabulary and context checks; weights are unchanged.


,Question,Top-K,Retrieved Book IDs,Visible Prompt,Prompt Characters,Any Evidence Truncated,Generated Response
0,Q1,1,[01_02],s.a. cosby\nking of ashes by,27,False,stems.\nlanguage models predict the next token...
1,Q1,2,"[01_02, 01_08]",mike ri s.a. co\nking of ashes by,32,True,stems.\nlanguage models predict the next token...
2,Q1,3,"[01_02, 01_08, 01_05]",u ma mike s.a. \nking of ashes by,32,True,stems.\nlanguage models predict the next token...
3,Q1,5,"[01_02, 01_08, 01_05, 04_10, 03_05]",ia al u mi s.a\nking of ashes by,32,True,stems.\nlanguage models predict the next token...
4,Q2,1,[02_01],susan choi\nflashlight by,24,False,uts th ton programming.\nstudents develop arti...
5,Q2,2,"[02_01, 02_08]",mike cur susan cho\nflashlight by,32,True,utse ursing probabilities.\nattention helps mo...
6,Q2,3,"[02_01, 02_08, 02_07]",elain mike susan \nflashlight by,32,True,utse ursing probabilities.\nattention helps mo...
7,Q2,5,"[02_01, 02_08, 02_07, 02_05, 03_05]",ia seb ela mik sus\nflashlight by,32,True,utse ursing probabilities.\nattention helps mo...
8,Q3,1,[03_03],laila lalami\nthe dream hotel by,31,False,utses use models and views.\ntraining changes ...
9,Q3,2,"[03_03, 05_08]",maggie laila \nthe dream hotel by,32,True,utses use models and views.\ntraining changes ...


## Compatibility verification

The fresh run passed strict checks for all **25 initial prompts** and every model forward pass. Each prompt round-trips through the saved tokenizer exactly, the baseline and RAG question cues match, and the model's weights remain equal to the loaded checkpoint.

| Condition | Prompts | Initial prompt characters | Maximum actual model input | Vocabulary valid for all prompts |
|---|---:|---:|---:|---|
| Without RAG | 5 | 8–18 | 32 | Yes |
| With RAG | 20 | 24–32 | 32 | Yes |

The tokenizer audit records lowercasing for every cue and author field used in prompts. It also found the following change beyond lowercasing:

| Retrieval document | Original author | Normalized author | Change |
|---|---|---|---|
| 01_05 | Uzma Jalaluddin | `u ma jalaluddin` | 'z' replaced with a space |

No new vocabulary entries were added. These substitutions can lose information: an unsupported letter is not learned by replacing it. Separately, all 15 Top-2/3/5 prompts deliberately shorten at least one normalized author, while every Top-1 prompt keeps the full requested author. The code records omitted text and character counts.

During generation, only the newest 32 characters are available at each step; after at most 32 generated characters, none of the initial retrieved text remains in the window.

In [5]:
baseline_audit = baseline_df[[
    "Question", "Visible Prompt", "Prompt Characters", "Vocabulary Valid",
    "Max Model Input Characters"
]].assign(Condition="Without RAG", **{"Top-K": 0})
rag_audit = rag_df[[
    "Question", "Top-K", "Visible Prompt", "Prompt Characters",
    "Vocabulary Valid", "Max Model Input Characters"
]].assign(Condition="With RAG")
prompt_audit_df = pd.concat([baseline_audit, rag_audit], ignore_index=True)

assert len(prompt_audit_df) == 25
assert prompt_audit_df["Vocabulary Valid"].all()
assert prompt_audit_df["Prompt Characters"].between(1, CONTEXT_LENGTH).all()
assert prompt_audit_df["Max Model Input Characters"].between(1, CONTEXT_LENGTH).all()
for row in prompt_audit_df.to_dict("records"):
    encode_slm_prompt(row["Visible Prompt"])
assert all(
    row["Question Cue"] == baseline_df.loc[
        baseline_df["Question"] == row["Question"], "Question Cue"
    ].iloc[0]
    for row in rag_df.to_dict("records")
)
assert all(torch.equal(checkpoint_weights[name], tensor)
           for name, tensor in model.state_dict().items())

compatibility_summary_df = prompt_audit_df.groupby("Condition", sort=False).agg(
    Prompts=("Question", "size"),
    Shortest_Prompt=("Prompt Characters", "min"),
    Longest_Prompt=("Prompt Characters", "max"),
    Max_Model_Input=("Max Model Input Characters", "max"),
    All_Vocabulary_Valid=("Vocabulary Valid", "all"),
).reset_index()
display(compatibility_summary_df)

used_book_ids = {book_id for ids in rag_df["Retrieved Book IDs"] for book_id in ids}
tokenizer_rows = [
    {"source": row["Question"], "field": "question cue", **row["Question Normalization"]}
    for row in baseline_df.to_dict("records")
]
tokenizer_rows += [
    {"source": book["book_id"], "field": "retrieved author", **book["author_normalization"]}
    for book in book_records if book["book_id"] in used_book_ids
]
tokenizer_audit_df = pd.DataFrame(tokenizer_rows)
lossy_normalization_df = tokenizer_audit_df[
    tokenizer_audit_df["unicode_changed"] | (tokenizer_audit_df["replacement_count"] > 0)
]
print("Tokenizer audit:", len(tokenizer_audit_df), "distinct cue/author fields used in prompts.")
if lossy_normalization_df.empty:
    print("These selected fields needed only lowercasing; no Unicode changes or unsupported replacements.")
else:
    display(lossy_normalization_df[[
        "source", "field", "original_text", "normalized_text",
        "unicode_changed", "unsupported_replaced", "replacement_count"
    ]])

# Ground truth is read only AFTER generation and is used only for evaluation.
expected_authors = {}
for question_id, question in BASELINE_QUESTIONS:
    requested_title = question.split("Who wrote ", 1)[1].rstrip(" ?")
    matches = [book for book in book_records if book["title"].casefold() == requested_title.casefold()]
    assert len(matches) == 1
    expected_authors[question_id] = matches[0]["author"]

def contains_expected_author(question_id, response):
    def letters(text):
        normalized, _ = normalize_for_slm(text)
        return "".join(char for char in normalized if char.isalpha())
    return letters(expected_authors[question_id]) in letters(response)

baseline_df["Correct Author Present"] = [
    contains_expected_author(row["Question"], row["SLM Response Without RAG"])
    for row in baseline_df.to_dict("records")
]
rag_df["Correct Author Present"] = [
    contains_expected_author(row["Question"], row["Generated Response"])
    for row in rag_df.to_dict("records")
]
changed_responses = sum(
    row["Generated Response"] != baseline_df.loc[
        baseline_df["Question"] == row["Question"], "SLM Response Without RAG"
    ].iloc[0] for row in rag_df.to_dict("records")
)
print("Correct author present:", int(baseline_df["Correct Author Present"].sum()),
      "of 5 baseline responses;", int(rag_df["Correct Author Present"].sum()), "of 20 RAG responses.")
print("PASS: every actual model input stayed within the checkpoint's context and vocabulary; weights unchanged.")

,Condition,Prompts,Shortest_Prompt,Longest_Prompt,Max_Model_Input,All_Vocabulary_Valid
0,Without RAG,5,8,18,32,True
1,With RAG,20,24,32,32,True


Tokenizer audit: 28 distinct cue/author fields used in prompts.


,source,field,original_text,normalized_text,unicode_changed,unsupported_replaced,replacement_count
6,01_05,retrieved author,Uzma Jalaluddin,u ma jalaluddin,False,['z'],1


Correct author present: 0 of 5 baseline responses; 0 of 20 RAG responses.
PASS: every actual model input stayed within the checkpoint's context and vocabulary; weights unchanged.


## Analysis

1. **Did providing retrieved information change the SLM's responses?** Yes: all 20 RAG continuations differed from the corresponding no-RAG responses. Both conditions used the same normalized compact question cue and generation settings, so retrieved evidence was the prompt difference.

2. **Did RAG improve factual accuracy?** No. Top-1 retrieved the intended book and supplied its complete normalized author for all five questions, but none of the 20 RAG responses named the requested author.

3. **Did retrieving more documents always improve the responses?** No: Top-2, Top-3, and Top-5 produced no correct author. All 15 of those prompts shortened at least one author and omitted the other retrieved books' title associations, which may contribute to failure.

4. **How did the SLM's context-length limitation affect the results?** The 32-character limit required a compact question cue and explicit evidence truncation; all actual inputs stayed within that limit, with exact surviving text shown in the tables. As generation continued, older evidence left the rolling window and could no longer affect subsequent predictions directly.

5. **Why might RAG behave differently with a small character-level model compared with a larger pretrained language model?** This SLM learned next-character patterns from a tiny, repetitive corpus, and its vocabulary cannot represent every new name without lossy substitutions such as replacing `z` in Uzma. A larger pretrained model may better interpret longer evidence and follow questions, although retrieval still does not guarantee correct answers.